In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [ ]:
# Read BBHI main data
m_df = pd.read_csv("~/Documents/2023:2024/Data/BBHI/BBHI Data Timept1 NPS.csv")

print(f"Number of participants: {len(m_df)}")

# Read BBHI education data
df_edu = pd.read_excel("~/Documents/2023:2024/Data/BBHI/BBHI_YoE.xlsx")

In [ ]:
# Calculate ages at baseline to filter df for 60+ yrs

def calculate_age(birth_date: str, collection_date: str) -> float:
    """Calculate exact age at baseline.

    Args:
        birth_date (str): Date of birth in format mm/dd/yyyy
        collection_date (str): Date of data collection in format mm/dd/yyyy

    Returns:
        age (float): Exact age at assessment
    """
    birth_date = datetime.strptime(birth_date, "%m/%d/%Y")
    collection_date = datetime.strptime(collection_date, "%m/%d/%Y")

    # Calculate difference in days
    days_difference = (collection_date - birth_date).days

    # Convert days to years using the average number of days in a year 
    age = days_difference / 365.25

    return age

# Assume m_df is your DataFrame
# 'Q1 GNRL Age' is DOB and 'w1_nps_date' is data collection date
# Add a new column to the DataFrame with exact age
m_df["w1_age"] = m_df.apply(
    lambda row: calculate_age(row["Q1 GNRL Age"], row["w1_nps_date"]), axis=1
)

# Manually check that everything is running correctly
print(m_df[["id", "w1_age", "Q1 GNRL Age", "w1_nps_date"]].head(5))

In [ ]:
# Filter df to only include 60+yrs

# Print the number of participants before filtering
print(f"Number of participants before filtering: {len(m_df)}")

agefiltered_df = m_df[m_df["w1_age"] >= 60]

# Print the number of participants after filtering
print(f"Number of participants after filtering: {len(agefiltered_df)}")

agefiltered_df[["id", "w1_age"]].head(5)

In [ ]:
# Read BBHI MRI data timept 1
df_MRI1 = pd.read_csv("~/Documents/2023:2024/Data/BBHI/MRI_info_TP1.csv")

# Create a new variable to see who has MRI data
mask = (
    (df_MRI1["mri_1_T1_discarded"] != "discarded")
    & (df_MRI1["mri_1_T2_discarded"] != "discarded")
    & (df_MRI1["mri_1_T1_availability"] > 0)
    & (df_MRI1["mri_1_T2_availability"] > 0)
    & (df_MRI1["mri_1_resting_duration"] == 10)  # This is because some of the first MRI scans were shorter and need to be excluded from tp1
    & (df_MRI1["mri_1_resting_availability"] == 1)
)

df_MRI1["mri_data"] = np.where(mask, 1, 0)

df_MRI1.sample(10)[
    [
        "mri_1_T1_discarded",
        "mri_1_T2_discarded",
        "mri_1_T2_availability",
        "mri_1_resting_duration",
        "mri_1_T1_availability",
        "mri_1_resting_availability",
        "mri_data",
    ]
]

In [ ]:
# Read BBHI MRI data timept 2
df_MRI2 = pd.read_csv("~/Documents/2023:2024/Data/BBHI/MRI_info_TP2.csv")

# Remove the sub- in the id (id numbers are formatted with sub-number in this csv only)
df_MRI2["id"] = df_MRI2["id"].str.replace("sub-", "").astype(int)

# Create a new variable to see who has MRI data
mask = (
    (df_MRI2["mri_2_T1_discarded"] != "discarded")
    & (df_MRI2["mri_2_T2_discarded"] != "discarded")
    & (df_MRI2["mri_2_T1_availability"] > 0)
    & (df_MRI2["mri_2_T2_availability"] > 0)
    & (df_MRI2["mri_2_resting_availability"] == 1)
)

df_MRI2["mri_data_tp2"] = np.where(mask, 1, 0)

df_MRI2.sample(10)[
    [
        "mri_2_T1_discarded",
        "mri_2_T2_discarded",
        "mri_2_T1_availability",
        "mri_2_T2_availability",
        "mri_2_resting_availability",
        "mri_data_tp2",
    ]
]

In [ ]:
# Merge dfs by participant ID for main data and education
merged_df = pd.merge(agefiltered_df, df_edu, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(merged_df)}")

In [ ]:
# Merge dfs by participant ID for main data and MRI timept 1
merged_df = pd.merge(merged_df, df_MRI1, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(merged_df)}")

In [ ]:
# Merge dfs by participant ID for main data and MRI timept 2
merged_df = pd.merge(merged_df, df_MRI2, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(merged_df)}")

In [ ]:
# Filter df to only include participants with MRI data

# Print the number of participants before filtering
print(f"Number of participants before filtering: {len(merged_df)}")

filtered_df = merged_df[(merged_df["mri_data"] == 1) & (merged_df["mri_data_tp2"] == 1)]

# Print the number of participants after filtering
print(f"Number of participants after filtering: {len(filtered_df)}")

filtered_df[["id", "mri_data", "mri_data_tp2"]].head(5)

In [ ]:
# Read BBHI timepoint 2 neuropsych data
np_tp2_df = pd.read_csv("~/Documents/2023:2024/Data/BBHI/BBHI Data Timept2 NPS.csv")

print(f"Number of participants: {len(np_tp2_df)}")

In [ ]:
# Rename tp2 variables 
def add_prefix_to_selected_columns(df, columns_to_prefix, prefix="w2_"):
    """
    Add a prefix to specific column names in the DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame whose column names will be updated.
        columns_to_prefix (list of str): The list of column names to which the prefix will be added.
        prefix (str): The prefix to add to each specified column name. Default is 'w1_'.
    
    Returns:
        pd.DataFrame: DataFrame with updated column names.
    """
    # Create a dictionary for renaming columns only in the specified list
    new_column_names = {col: prefix + col for col in df.columns if col in columns_to_prefix}

    # Use rename method to update the DataFrame with the new column names
    df = df.rename(columns=new_column_names)
    return df

# List of variables 
columns_to_update = [    
    "delayed_recall_raw",  # RAVLT tp2 
    "sem_fluency_raw",  # Semantic fluency tp2 
    "tmt_b_raw",  # TMT B tp2 
    "tmt_a_raw",  # TMT A tp2 
    "direct_digits_raw",  # Digit span forward tp2 
    "inverse_digits_raw"
    ]

np_tp2_df = add_prefix_to_selected_columns(np_tp2_df, columns_to_update)

np_tp2_df[['id', 'w2_sem_fluency_raw', 'w2_tmt_b_raw', 'w2_tmt_a_raw', 'w2_direct_digits_raw', 'w2_inverse_digits_raw']].head(5)

In [ ]:
# Merge dfs by participant ID for neuropsych tp2 data and filtered df
filtered_df = pd.merge(filtered_df, np_tp2_df, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(filtered_df)}")

filtered_df[["id", "w1_age"]].head(5)

In [ ]:
# Calculate ages at tp2 assessment

# 'Q1 GNRL Age' is DOB and 'nps_date' is data collection date
filtered_df["w2_age"] = filtered_df.apply(
    lambda row: calculate_age(row["Q1 GNRL Age"], row["nps_date"]), axis=1
)

# manually check that everything is running correctly
filtered_df[["id", "w2_age", "Q1 GNRL Age", "nps_date"]].head(5)

In [ ]:
# Drop participants with missing neuropsych data and -1 values in the specific columns

# Get the number of rows before dropping participants
rows_before = filtered_df.shape[0]

columns = [
    "w1_age",  # Age tp1
    "w2_age",  # Age tp2
    "YoE",  # Years of education
    "w1_delayed_recall_raw",  # RAVLT tp1
    "w1_sem_fluency_raw",  # Semantic fluency tp1
    "w1_tmt_b_raw",  # TMT B tp1
    "w1_tmt_a_raw",  # TMT A tp1
    "w1_direct_digits_raw",  # Digit span forward tp1
    "w1_inverse_digits_raw",  # Digit span backward tp1
    "w2_delayed_recall_raw",  # RAVLT tp2 
    "w2_sem_fluency_raw",  # Semantic fluency tp2 
    "w2_tmt_b_raw",  # TMT B tp2 
    "w2_tmt_a_raw",  # TMT A tp2 
    "w2_direct_digits_raw",  # Digit span forward tp2 
    "w2_inverse_digits_raw",  # Digit span backward tp2 
]

# Drop participants with missing neuropsych data
filtered_df.dropna(subset=columns, inplace=True)

# Drop participants with -1 values in specific columns
for col in columns:
    filtered_df = filtered_df[filtered_df[col] != -1]

# Get the number of rows after dropping participants
rows_after = filtered_df.shape[0]

# Calculate the number of participants dropped
participants_dropped = rows_before - rows_after

participants_dropped


In [ ]:
# Rename the sex column 
filtered_df = filtered_df.rename(columns={'Q1 GNRL Sex': 'sex'})  

# Replace 1 with 'male' and 2 with 'female' in the sex column
filtered_df['sex'] = filtered_df['sex'].replace({1: 'male', 2: 'female'})

In [ ]:
# Filter df to only include needed columns  
filtered_df = filtered_df[[
    "id",
    "w1_age",
    "YoE",
    "sex",
    "w1_delayed_recall_raw",
    "w1_tmt_b_raw",
    "w1_sem_fluency_raw",
    "w1_tmt_a_raw",
    "w1_direct_digits_raw",
    "w1_inverse_digits_raw",
    "w2_age", 
    "w2_delayed_recall_raw", 
    "w2_sem_fluency_raw",
    "w2_tmt_b_raw", 
    "w2_tmt_a_raw", 
    "w2_direct_digits_raw", 
    "w2_inverse_digits_raw"
]]

In [ ]:
# Look at data for available longitudinal analysis

participants = filtered_df["w1_age"].count()
mean_age = filtered_df["w1_age"].mean()
std_dev_age = filtered_df["w1_age"].std()
min_age = filtered_df["w1_age"].min()
max_age = filtered_df["w1_age"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")

In [ ]:
# Export this df to a csv to use for future analysis

filtered_df.to_csv(
    "/Users/rachelmorse/Documents/2023:2024/Data/Exported data/clean_bbhi.csv", index=False
)